# M²S²L: Mamba-based Multi-Scale Spatial-temporal  Learning for Video Anomaly Detection

M²S²L is an **unsupervised reconstruction-based Video Anomaly Detection (VAD)** framework built upon **Mamba**. The main motivation is that anomalies can occur at **different spatial scales**, **different temporal durations**, and affect **appearance** and **motion** differently. Previous Mamba-based (VADMamba and STNMamba) approaches typically employ homogeneous encoder architectures that process both appearance and motion information using identical structures, failing to capture the distinct characteristics of visual appearance and temporal motion patterns.

To address these limitations, we propose the Mamba-based multi-scale spatial-temporal learning (M2S2L) for VAD in this paper. 

- Introduces specialized dual-stream Mamba encoders that separately model appearance and motion normality patterns, enabling task-specific feature learning while preserving computational efficiency. 
- Additionally, we develop hierarchical multi-scale processing mechanisms that capture spatial-temporal patterns across different granularities, from fine-grained local anomalies to coarse-grained scenelevel irregularities.

## Architecture 

<div>
	<img src="images/M2S2Mamba.png" width="1000">
</div>

The system consists of three core components: 
1. **Multi-scale spatial normality learning** for hierarchical appearance pattern modeling
2. **Multi-scale temporal normality learning** for motion dynamics across different time scales
3. **Feature decomposition** with specialized decoding for taskspecific reconstruction.

Given input video clips $V_{t-k:t} = \{V_{t-k}, V_{t-k+1}, \ldots, V_t\}\in \mathbb{R}^{k \times H \times W \times 3}$ where $k$ denotes the temporal window length, the **M2S2L framework simultaneously learns appearance and motion normality through parallel processing streams**.

During **training**, the framework **exclusively** observes **normal sequences** to establish baseline patterns. At inference time, **anomalous** events are identified **through reconstruction discrepancies in both modalities**.

## Components

### 1. Multi-Scale Spatial Normality Learning

Captures appearance patterns across multiple spatial granularities to handle anomalies that manifest at different scales. The hierarchical spatial processing employs **three parallel encoders** operating at distinct patch resolutions: fine, medium, and coarse.

Input sequences are decomposed into multi-granularity patch representations:

\begin{equation*}
P(i) = \operatorname{PatchPartition}(V_{t-k:t}, r_i),
\quad \text{for } i \in \{1, 2, 3\},
\quad \text{where } r_i \in \{4, 8, 16\}
\end{equation*}

correspond to fine, medium, and coarse spatial resolutions respectively.

Each granularity captures complementary spatial information: 

- **Fine scale patches** preserve detailed texture information for detecting subtle appearance anomalies

\begin{equation*}
P(1) \in \mathbb{R}^{k \times (H/4 \times W/4) \times C}
\end{equation*}

- **Medium-scale patches** balance detail preservation with computational efficiency

\begin{equation*}
P(2) \in \mathbb{R}^{k \times (H/8 \times W/8) \times C}
\end{equation*}

- **Coarsescale patches** capture global scene context for understanding large-scale structural changes.

\begin{equation*}
P(3) \in \mathbb{R}^{k \times (H/16 \times W/16) \times C}
\end{equation*}

Each spatial scale is processed through Mamba encoders utilizing Multi-Scale Visual State Space Blocks (MS-VSSB), as shown in previous figure, hence:


\begin{equation*}
X^{(i)} = \sum_{j \in \{1,3,5\}} \operatorname{DWConv}_{j \times j}(P^{(i)}),
\tag{1}
\end{equation*}

\begin{equation*}
G^{(i)} = \operatorname{SpatialEncoder}_i\left(X^{(i)} + P^{(i)}\right).
\tag{2}
\end{equation*}

>Note: The SpatialEncoder is not described, however from the figure, the MS-VSSB, we can see that the operation is the VSSB, like in STNMamba. Also de linear layer is omitted in the equation.

<div>
	<img src="images/M2S2MambaB.png" width="300">
</div>

where $G^{(i)} \in \mathbb{R}^{K \times N_i \times D}$ represents the spatial features at scale $i$, and $N_i$ denotes the spatial sequence length at each granularity. 

Spatial features from different scales are integrated through **adaptive importance weighting**:

\begin{equation*}
\beta_i = \mathrm{MLP}(\mathrm{GlobalPool}(G^{(i)})), \quad i \in \{1, 2, 3\}.
\end{equation*}

The final spatial representation is obtained:

\begin{equation*}
G_{\mathrm{spatial}} =
\sum_{i=1}^{3}
\mathrm{Softmax}([\beta_1, \beta_2, \beta_3])_i
\cdot
\mathrm{Resize}(G^{(i)}, N_1),
\tag{3}
\end{equation*}

where Resize operations standardize different scales to unified spatial dimensions, yielding $G_{\mathrm{spatial}} \in \mathbb{R}^{K \times N_1 \times D}$.

### STNMamba vs. M²S²Mamba

| Aspect | STNMamba | M²S²Mamba |
|---|---|---|
| Patch partition | 4×4 Patch Embedding. | Three granularities: 4×4, 8×8, and 16×16. |
| Spatial modeling | Hierarchical architecture with multiple processing stages and MS-VSSB. | Spatial encoders operating at different granularities. |
| Multi-scale | Parallel DWConv operations with 1×1, 3×3, and 5×5 kernels. | DWConv operations with 1×1, 3×3, and 5×5 kernels at each granularity. |
| Feature fusion | Hierarchical feature processing. | Adaptive weighting using MLP and Softmax to integrate multi-scale features. |
| Temporal modeling | Spatio-temporal modeling using Mamba. | Multi-temporal encoders to capture motion dynamics at different temporal scales. |

**Main difference:**

- **STNMamba:** Employs a hierarchical spatial architecture starting with a 4×4 Patch Embedding and uses multi-kernel convolutions to enrich spatial features.
- **M²S²Mamba:** Extends spatial processing through three explicit patch granularities (4×4, 8×8, and 16×16), integrating their features through adaptive weighting. It also incorporates multi-scale temporal modeling.

**In summary:** M²S²Mamba emphasizes multi-granularity spatial representations and adaptive feature fusion, alongside temporal modeling at different scales.

*Hierarchy refers to organizing feature representations into successive stages, where spatial resolution is progressively reduced and features become more abstract (e.g., 56×56 → 28×28 → 14×14).*

### Multi-Scale Temporal Normality Learning

Models motion dynamics across varying time horizons to capture anomalous behaviors that occur over different temporal durations. 

Three temporal windows are constructed to analyze motion patterns at multiple scales: 

- **Short-term** clips $V_{t−3:t}$ for instantaneous motion changes
- **Medium-term** clips $V_{t−7:t}$ for action-level dynamics
- **Long-term** clips $V_{t−15:t}$ for behavioral trend analysis.

For computational efficiency, temporal motion is represented through **frame differencing** rather than optical flow computation: 

\begin{equation*}
D_{t−w_j:t−1} = V_{t−w_j+1:t} − V_{t−w_j:t−1}
\end{equation*}

where $w_j$ denotes  the window size for temporal scale $j$.

Each temporal scale employs specialized **Temporal Mamba Blocks (TMB)** optimized for motion sequence modeling. TMB incorporates temporal positional encoding and scale-specific parameterization:

\begin{align}
A_j &= -\exp(\delta_j)\odot\Lambda_j, \tag{4}\\
B_j &= \delta_j\odot\sigma\left(\operatorname{Linear}(D_{t-w_j:t-1})\right), \tag{5}\\
C_j &= \sigma\left(\operatorname{Linear}(D_{t-w_j:t-1})\right). \tag{6}
\end{align}

where $\delta_j$ represents scale-specific step parameters, $\Lambda_j$ denotes diagonal state matrices, and $\sigma$ is the activation function.

> **Historical background of the Temporal Mamba Block (TMB):** M²S²L (Liu et al., 2025) introduces a multi-scale temporal modeling block based on selective state-space modeling. Its parameterization follows the general principles of Mamba (Gu & Dao, 2024), while introducing scale-indexed parameters \(A_j\), \(B_j\), and \(C_j\), a temporal window \(w_j\), and sigmoid-based projections. These equations define the scale-specific state-space dynamics used by the TMB. Their exact correspondence to Mamba's original parameterization and the novelty of the individual modifications must be distinguished from the overall multi-scale design.

Temporal features are computed by integrating motion differences with positional information: $H(j) = \text{TemporalEncoder}_j (D_{t−w_j:t−1} + \text{PE}_{t−w_j:t−1})$, where PE represents temporal positional encoding. Multi-temporal features $H_{\text{temporal}}$ are fused through attention-based aggregation that automatically determines the relevance of each temporal scale.

### 3. Feature Decomposition and Decoding

<div>
	<img src="images/MS2SMambaF.png" width="150">
</div>

After combining the spatial and temporal features

\begin{equation*}
F_{\text{fused}} = \text{LN}(G_\text{spatial} + H_\text{temporal})
\end{equation*}

M²S²L decomposes the fused representation into task-common and task-specific components through feature disentanglement.

> *Feature disentanglement is the process of separating different types of information into distinct features, so that each feature captures a specific aspect of the input.*


**Task-common**

- Common features ($F_{common}$)

**Task-specific**

- Appearance-specific features ($F_{app}$)
- Motion-specific features ($F_{motion}$)


Three memory banks $\{M_c, M_a, M_m\}$ corresponding to common, appearance, and motion features respectively are employed to record normal prototypes through reading and writing operations during training.


Specialized decoders then process the decomposed features, with the appearance decoder combining common and appearance specific features to generate frame predictions $\hat{V}_{t+1}$, while the motion decoder fuses common and motion-specific features to reconstruct motion fields $\hat{M}_{t}$.

## Loss function

Training is guided by a loss function balancing appearance reconstruction, motion modeling, and feature specialization:

\begin{equation*}
\mathcal{L}_{total} = \mathcal{L}_{frame} + \lambda_m \mathcal{L}_{motion} + \lambda_s \mathcal{L}_{separate}
\end{equation*}

where the frame prediction loss combines pixel-level and gradient consistency:

\begin{equation*}
\mathcal{L}_{frame} =
\left\| \hat{V}_{t+1} - V_{t+1} \right\|_2^2
+ \lambda_g
\left\| \nabla \hat{V}_{t+1} - \nabla V_{t+1} \right\|_1
\tag{7}
\end{equation*}

The motion loss ensures accurate dynamics modeling:

\begin{equation*}
\mathcal{L}_{motion} =
\left\| \hat{M}_t - M_t \right\|_2^2
+ \lambda_{ssim}
\left(1 - SSIM(\hat{M}_t, M_t)\right)
\tag{8}
\end{equation*}

The separation loss promotes task specialization:

\begin{equation*}
\mathcal{L}_{separate} =
- CosineSim(F_{app}, F_{motion})
+ \left\| F_{app}F_{motion}^{T} \right\|_F^2
\tag{9}
\end{equation*}

## Anomaly Scoring

The final anomaly score combines the reconstruction quality of both appearance and motion.

The procedure is:

1. Compute frame reconstruction error.
2. Compute motion reconstruction error.
3. Convert both errors into **PSNR** values.

\begin{equation*}
PSNR_{frame} = 10 \log_{10} \left( \frac{MAX^2}{MSE(\hat{V}_{t+1}, V_{t+1})} \right)
\end{equation*}

\begin{equation*}
PSNR_{motion} = 10 \log_{10} \left( \frac{MAX^2}{MSE(\hat{M}_t, M_t)} \right)
\end{equation*}

4. Compute a weighted combination:

$$
PSNR_{combined}
=
\alpha PSNR_{frame}
+
(1-\alpha)PSNR_{motion}
$$

where **α** balances the contribution of appearance and motion.

Finally, anomaly scores are obtained through min-max normalization across the entire test sequence.


## Results

| Method    |     Ped2 |   Avenue | ShanghaiTech |    FLOPs |   Params |    FPS |
| --------- | -------: | -------: | -----------: | -------: | -------: | -----: |
| VADMamba  | **98.5** |     91.5 |         77.0 |        — |    28.1M | **90** |
| STNMamba  |     98.0 |     89.0 |         74.9 | **1.5G** | **7.2M** |     40 |
| **M²S²L** | **98.5** | **92.1** |     **77.9** |    20.1G |    14.9M |     45 |

**Main observations**

- **Highest AUC on Avenue (92.1%)** and **ShanghaiTech (77.9%)** among the compared Mamba-based methods.
- Matches **VADMamba** on **UCSD Ped2 (98.5%)** while outperforming it on the more challenging datasets.
- Compared with **STNMamba**, the gains are:
    - **+0.5%** on Ped2
    - **+3.1%** on Avenue
    - **+3.0%** on ShanghaiTech
- Although it is not the fastest model, **45** FPS is still sufficient for real-time inference while achieving better detection accuracy.

---

## Key Idea

Unlike previous Mamba-based VAD methods, M²S²L explicitly specializes feature learning along **two dimensions**:

- **Scale specialization:** captures anomalies ranging from small local changes to large scene-level events.
- **Modality specialization:** independently models appearance and motion while still leveraging their shared information.

This design enables richer spatial-temporal representations while preserving the computational efficiency of Mamba, resulting in improved detection performance, particularly on challenging datasets such as **CUHK Avenue** and **ShanghaiTech**.

---

## Comparison with previous models

| Modelo       | Spatial branch    | Temporal branch    | Feature decomposition | Multi-scale             |
| ------------ | ----------------- | ------------------ | --------------------- | ----------------------- |
| **VADMamba** | ❌ (una sola rama) | ✅ (Mamba temporal) | ❌                     | ❌                       |
| **STNMamba** | ✅                 | ✅                  | ❌                     | ❌                       |
| **M²S²L**    | ✅                 | ✅                  | ✅                     | ✅ (espacial + temporal) |
